In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import celltypist
from celltypist import models
from celltypist.models import Model

# import data

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_01.h5ad")
adata_scraw=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")

In [ ]:
adata_scraw

In [ ]:
# preprocessing pipeline
sc.pp.normalize_total(adata_scraw)
sc.pp.log1p(adata_scraw)
adata_scraw.raw = adata_scraw.copy()

sc.pp.highly_variable_genes(adata_scraw)
adata_scraw = adata_scraw[:, adata_scraw.var['highly_variable']]

sc.pp.scale(adata_scraw, max_value=10)
sc.tl.pca(adata_scraw, svd_solver='arpack')

In [ ]:
# t-sne 
sc.tl.tsne(adata_scraw, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_scraw, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_scraw)
sc.pl.umap(adata_scraw, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_scraw.obsm['X_pca'],adata_scraw.obs, 'sample')

# Harmony 결과를 저장
adata_scraw.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_scraw, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_scraw)
sc.pl.umap(adata_scraw, color=['state'])

In [ ]:
top_genes_filtered = [g for g in top_genes_state if g in adata_scraw.var_names]

sc.pl.heatmap(
    adata_scraw,
    var_names=top_genes_filtered,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    figsize=(10, 7)
)

In [ ]:
missing_genes = [gene for gene in top_genes_state if gene not in adata_scraw.var_names]
print(f"❌ Missing genes: {missing_genes}")

In [ ]:
rank_df_state

In [ ]:
rank_df_state.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/01_Data/scTotal_State_DEG.csv")

In [ ]:
adata_scraw.obs

In [ ]:
adata_scraw.obs['celltypist_liver'] = adata_scrna.obs['celltypist_liver']

In [ ]:
adata_scraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")
sc.pp.normalize_total(adata_scraw, target_sum=1e4)
sc.pp.log1p(adata_scraw)
adata_scrna.raw = adata_scraw

In [ ]:
sc.tl.leiden(adata_scrna, resolution=0.5, key_added="leiden")
sc.pl.umap(adata_scrna, color=['leiden'])

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='leiden',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
predictions_cluster = celltypist.annotate(adata_scrna, model='Healthy_Human_Liver.pkl', majority_voting=True, over_clustering=adata_scrna.obs["leiden"]
)
adata_scrna =predictions_cluster.to_adata()

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='majority_voting',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
adata_scrna.obs['celltype'] = adata_scrna.obs['majority_voting'] 

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='celltypist_liver',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='PTPRC',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='CD45+',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='SLC7A5',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='SLC7A5 expression',
    size=10
)

In [ ]:
sc.pl.umap(adata_scrna[adata_scrna.obs['state'] == 'healthy'], color='celltype', title='Healthy')
sc.pl.umap(adata_scrna[adata_scrna.obs['state'] == 'NAFLD'], color='celltype', title='MASLD')

In [ ]:
sc.pl.umap(adata_scrna[adata_scrna.obs['state'] == 'healthy'], color='SLC7A5', title='Healthy')
sc.pl.umap(adata_scrna[adata_scrna.obs['state'] == 'NAFLD'], color='SLC7A5', title='MASLD')

In [ ]:
adata_scrna.obs['state'].value_counts()

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_scrna.obs['celltype']
batch_labels = adata_scrna.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
adata_scrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_02.h5ad")

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_02.h5ad")

In [ ]:
adata_scraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")
adata_scraw.obs['celltype']=adata_scrna.obs['celltype']

In [ ]:
adata_T = adata_scrna[adata_scrna.obs['celltype'].isin(['T cells'])].copy()
adata_T

In [ ]:
adata_scrna.obs

In [ ]:
sc.tl.dendrogram(adata_scrna,groupby='state')
sc.tl.rank_genes_groups(adata_scrna, groupby='state', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scrna, n_genes=10)

rank_df_state = sc.get.rank_genes_groups_df(adata_scrna, group=None)
top_genes = rank_df_state.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scrna,
    var_names=top_genes,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df_state

In [ ]:
rank_df_state.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/01_Data/scTotal_state_DEG.csv")

# scRNA _ mono/MAC analysis

In [ ]:
adata_scraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")

In [ ]:
adata_scraw.obs['celltype']=adata_scrna.obs['celltype']

In [ ]:
adata_scraw.obs['celltype'].value_counts()

In [ ]:
adata_scmono = adata_scraw[adata_scraw.obs['celltype'].isin(['Macrophages','Mono+mono derived cells'])].copy()
adata_scmono

In [ ]:
adata_scmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono01.h5ad")

In [ ]:
adata_scmono=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono01.h5ad")

In [ ]:
adata_scmono.raw = adata_scmono

In [ ]:
sc.pp.normalize_total(adata_scmono, target_sum=1e4)
sc.pp.log1p(adata_scmono)
sc.pp.highly_variable_genes(adata_scmono, flavor='seurat', n_top_genes=2000)

In [ ]:
# Scale data
sc.pp.scale(adata_scmono, max_value=10)
# Run PCA
sc.tl.pca(adata_scmono, svd_solver='arpack')
print(adata_scmono.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_scmono, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_scmono.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_scmono, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_scmono, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_scmono)
sc.pl.umap(adata_scmono, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_scmono.obsm['X_pca'],adata_scmono.obs, 'sample')

# Harmony 결과를 저장
adata_scmono.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_scmono, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_scmono)

In [ ]:
sc.pl.umap(adata_scmono,color='state')

In [ ]:
sc.tl.rank_genes_groups(
    adata_scmono,
    groupby="state",        # 비교할 obs 열
    groups=["NAFLD"],     # 비교할 그룹
    reference="healthy",     # 참조 그룹
    use_raw=True,
    method="wilcoxon"       # 또는 't-test', 'logreg'
)

In [ ]:
deg_df = sc.get.rank_genes_groups_df(adata_scmono, group="NAFLD")
print(deg_df.head())

In [ ]:
filtered = deg_df[
    (deg_df['logfoldchanges'] > 0.58) &
    (deg_df['pvals_adj'] < 0.05)
]

In [ ]:
top20_filtered = filtered.sort_values(by='scores', ascending=False).head(50)
print(top20_filtered)

In [ ]:
deg_df['-log10(padj)'] = -np.log10(deg_df['pvals_adj'])
genes_of_interest = ["SLC7A5"]

plt.figure(figsize=(7, 6))
plt.scatter(
    deg_df['logfoldchanges'], 
    deg_df['-log10(padj)'], 
    c="grey", s=10, alpha=0.7
)

# 유의미한 cutoff 설정 (예: logFC > 1, padj < 0.05)
sig_up = (deg_df['logfoldchanges'] > 1) & (deg_df['pvals_adj'] < 0.05)
sig_down = (deg_df['logfoldchanges'] < -1) & (deg_df['pvals_adj'] < 0.05)

plt.scatter(deg_df.loc[sig_up, 'logfoldchanges'],
            deg_df.loc[sig_up, '-log10(padj)'],
            c="red", s=10, label="Upregulated")

plt.scatter(deg_df.loc[sig_down, 'logfoldchanges'],
            deg_df.loc[sig_down, '-log10(padj)'],
            c="blue", s=10, label="Downregulated")

# cutoff 선
plt.axvline(1, color="black", linestyle="--")
plt.axvline(-1, color="black", linestyle="--")
plt.axhline(-np.log10(0.05), color="black", linestyle="--")

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 Adjusted p-value")
plt.title("Volcano Plot: MASLD vs Normal")
plt.legend()

for gene in genes_of_interest:
    if gene in deg_df['names'].values:
        x = deg_df.loc[deg_df['names'] == gene, 'logfoldchanges'].values[0]
        y = deg_df.loc[deg_df['names'] == gene, '-log10(padj)'].values[0]
        plt.text(x, y, gene, fontsize=8, color="black")  # 텍스트 표시


plt.tight_layout()
plt.show()

In [ ]:
deg_df.to_csv("/home/jaehyunchoi/MASLD_collaboration/MP_analysis/DEG/scRNA_MASLD_DEG_markers.csv")

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='state',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='state',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='SLC7A5',
    #legend_loc='on data',
    legend_fontsize=10,
    frameon=False,
    title='SLC7A5',
    size=10
)

In [ ]:
adata_scmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")

In [ ]:
gene_sets = {
    "cDC1_genes": ['XCR1', 'CLNK', 'TACSTD2', 'CLEC9A', 'EGLN3', 'IDO1', 'DNASE1L3', 'CADM1', 'S100B', 'KIAA0226L', 'WDFY4', 'CCND1', 'DUSP4', 'P2RY14', 'C1orf54', 'NRARP', 'CPNE3', 'SLAMF7', 'GPR157', 'B3GNT7', 'FLT3', 'NET1', 'RGCC', 'SERPINF1', 'BATF3', 'NDRG2', 'SNX3', 'ADAM28', 'AIM2', 'IL18R1', 'GPR171', 'PTPRCAP', 'IRF8', 'CPVL', 'C12orf45', 'TPD52', 'SLC38A1', 'LMNA', 'CD59', 'C15orf48', 'NFATC2', 'NAV1', 'MAP2K6', 'PTMS', 'ARHGAP5', 'HLA-DPB1', 'CST7', 'LGALS2', 'HLA-DQB1', 'PPA1', 'TCTN3', 'APOL1', 'CAMK2D', 'HLA-DPA1', 'CD74', 'HLA-DQA1', 'CST3', 'BCL2L11', 'DHRS3', 'ID2', 'RAB11FIP1', 'QPRT', 'OSBPL9', 'PTPN22', 'PTPN7', 'FAM129A', 'DST', 'FUCA1', 'ANPEP', 'MARCKSL1', 'HLA-DRB1', 'CLIC2', 'STK17A', 'RGS1', 'HLA-DRA', 'BASP1', 'ACTN1', 'PDLIM7', 'CD40', 'AC093673.5', 'NCOA7', 'PDE4DIP', 'TUBA1C', 'STMN1', 'CSF2RA', 'CD38', 'MIR4435-1HG', 'DUSP2', 'LGMN', 'PKIB', 'VMO1', 'SPATS2L'],
    "cDC2_genes": ['FCER1A', 'PPP1R14A', 'CD1C', 'AKR1C3', 'ENHO', 'CD1E', 'SLC38A1', 'P2RY14', 'AFF3', 'FCGR2B', 'CLEC10A', 'NDRG2', 'CST7', 'PKIB', 'CACNA2D3', 'ADAM28', 'SDPR', 'PLD4', 'NET1', 'FLT3', 'GAS6', 'ALOX5AP', 'AGPAT9', 'HLA-DQA1', 'RGS1', 'FILIP1L', 'C12orf45', 'HLA-DQB1', 'IL1R2', 'ARHGAP5', 'HLA-DPB1', 'GPR183', 'AXL', 'ADAM8', 'USP53', 'SERPINF1', 'RGCC', 'HLA-DPA1', 'CD72', 'GSN', 'CLIC2', 'ACAP1', 'ATP1B1', 'HLA-DRA', 'PMAIP1', 'CD74', 'CSF2RA', 'AREG', 'PPA1', 'FPR3', 'HLA-DRB1', 'ALCAM', 'SPATS2L', 'ARL4C', 'PTMS', 'STMN1', 'LMNA', 'VSIG4'],
    "pDC_genes":['SCT', 'LRRC26', 'LILRA4', 'MAP1A', 'PTCRA', 'LAMP5', 'CLEC4C', 'MYBL2', 'SMIM5', 'KCNK17', 'PACSIN1', 'RP11-117D22.2', 'TCL1A', 'IL3RA', 'SPIB', 'PLD4', 'BCL11A', 'RASD1', 'FAM129C', 'TPM2', 'TSPAN13', 'PTPRS', 'SERPINF1', 'PPP1R14A', 'DERL3', 'SMPD3', 'IRF4', 'ITM2C', 'PFKFB2', 'PTGDS', 'FUT7', 'SOX4', 'IRF7', 'CXCR3', 'P2RY14', 'NOTCH4', 'PMEPA1', 'LINC00996', 'PPP1R14B', 'CLN8', 'VIPR2', 'AEBP1', 'C1orf186', 'MZB1', 'IRF8', 'TCF4', 'UGCG', 'GZMB', 'TNFRSF21', 'GAPT'],
    "MigcDC_genes": ['LAD1', 'CCR7', 'LAMP3', 'CCL19', 'CCL22', 'IL7R', 'SLC7A11', 'FSCN1', 'RAMP1', 'IDO1', 'CCL17', 'CD1B', 'MIR155HG', 'TNFRSF4', 'CXCL9', 'GPR157', 'DUSP4', 'IL4I1', 'EBI3', 'TRAF1', 'MARCKSL1', 'SPIB', 'ARHGAP10', 'HLA-DQB2', 'CD1E', 'MGLL', 'IL32', 'DUSP5', 'NRP2', 'CST7', 'PSD3', 'BIRC3', 'IRF4', 'C15orf48', 'HILPDA', 'GADD45A', 'SLC38A1', 'TRAF5', 'C12orf45', 'TXN', 'RAB9A', 'TBC1D4', 'TUBA1C', 'RGS1', 'SERPINF1', 'NMRK1', 'PPA1', 'CD40', 'GSN', 'POGLUT1', 'CLIC2', 'NAV1', 'PVRL2', 'PTGIR', 'RP11-138A9.2', 'FABP5', 'ALCAM', 'FAM129A', 'NET1', 'CSF2RA', 'ANTXR2', 'UGCG', 'GPR137B', 'CD1C', 'ADAM8', 'RELB', 'PALLD', 'SLAMF7', 'ST3GAL6', 'RP11-356I2.4', 'GPR183', 'HLA-DQA1', 'NFKB1', 'FLT3', 'RP11-138A9.1', 'CERS6', 'DNASE1L3', 'P2RY8', 'CD83', 'MAFF', 'SESN1', 'CYB5A', 'PTMS', 'SDC2', 'CCL5', 'ATP1B1', 'ETV3', 'ATF5', 'HLA-DQB1', 'FILIP1L', 'PMAIP1', 'PIM3', 'ID2', 'MARCKS', 'EPSTI1', 'BHLHE40', 'SOCS1', 'BASP1', 'HLA-DPB1', 'CD74', 'APOC1', 'HLA-DRA', 'CD72', 'GBP1', 'NR4A3', 'HLA-DPA1', 'PLEK', 'LMNA', 'CLEC10A'],
    "Monocyte_genes":['S100A8', 'S100A12', 'S100A9', 'VCAN', 'CTB-61M7.2', 'THBS1', 'EREG', 'MGST1', 'RP11-1143G9.4', 'ASGR1', 'BST1', 'AQP9', 'FCN1', 'CDA', 'LYZ', 'SLC2A3', 'MEGF9', 'APOBEC3A', 'CD300E', 'CD36', 'RAB27A', 'CLEC4E', 'TMEM176A', 'SELL', 'LGALS2', 'CXCL2', 'ITGAM', 'CSF3R', 'G0S2', 'EGR1', 'SOCS3', 'CCDC69', 'CAPG', 'TREM1', 'SLC11A1', 'AREG'],
    "Patmono_genes":['LYPD2', 'CDKN1C', 'CKB', 'PTP4A3', 'CD79B', 'HES4', 'PPM1N', 'SYTL1', 'FCGR3A', 'LILRA3', 'RHOC', 'RP11-290F20.3', 'TESC', 'SPN', 'TCF7L2', 'TPPP3', 'SLC2A6', 'SLC44A2', 'RRAS', 'SNX18', 'OAS1', 'FAM110A', 'VMO1', 'MTSS1', 'LILRB2', 'PHF19', 'TBC1D10C', 'FPR2', 'LILRA5', 'IFITM2', 'SIGLEC10', 'MYO1G', 'PAG1', 'SIDT2', 'LTB', 'LST1', 'S1PR4', 'ITGAL', 'STXBP2', 'CNIH4', 'GCH1', 'IFITM3', 'KLF2'],
    "LAM_genes":['FN1', 'CST6', 'SPP1', 'MT1G', 'TREM2', 'GPNMB', 'SLC16A10', 'CD9', 'EMP1', 'APOC1', 'LMNA', 'C15orf48', 'CLEC5A', 'IL7R', 'FABP5', 'RNASE1', 'FBP1', 'SDS', 'RAB13', 'OLR1', 'APOE', 'RGCC', 'GPX3', 'TSC22D1', 'SLC39A8', 'CAPG', 'ACP5', 'MMP19', 'ALCAM', 'GSN', 'FPR3', 'PDE4DIP', 'MT1X', 'LHFPL2', 'SDC2', 'MITF', 'PLA2G7', 'LGALS3', 'PHLDA1', 'ANKRD28', 'AVPI1', 'PTMS', 'MARCO', 'B3GNT5', 'PLTP', 'GPR183', 'DAB2', 'CD59', 'RGS1', 'LGMN'],
    "MATLAM_genes":['TREM2', 'GPNMB', 'CD9', 'RNASE1', 'OTOA', 'SDS', 'A2M', 'APOC1', 'LGMN', 'FCGBP', 'FABP5', 'SLCO2B1', 'APOE', 'C1QC', 'PLA2G7', 'FPR3', 'ID3', 'DAB2', 'AXL', 'C1QB', 'PLTP', 'RGS1', 'LTC4S', 'CD72', 'GATM', 'MGLL', 'LMNA', 'ABCC5', 'GAS6', 'ANKH', 'STMN1', 'C1QA', 'ALCAM', 'MSR1', 'PKIB', 'PHLDA1', 'ATP1B1', 'GPR34', 'PTMS', 'RGCC', 'MGAT4A', 'FCGR2B', 'LIPA', 'ACP5', 'CTSL', 'MITF', 'MAF', 'EBI3', 'KCNMA1', 'MERTK', 'VSIG4', 'C15orf48', 'FILIP1L', 'SEPP1', 'SLC40A1', 'CD81', 'GPR183', 'SDC2', 'GSN', 'FOLR2', 'CD59', 'CD84', 'CPM', 'MS4A4A', 'ABCA1', 'HLA-DQA1', 'CLEC10A', 'CLIC2', 'ARL4C', 'RGL1', 'EPB41L2', 'MMP19', 'AVPI1', 'ALB', 'C1orf54', 'PDK4', 'CD1C', 'VMO1'],
    "MAC1_genes":['IGSF21', 'FCGBP', 'USP53', 'ID3', 'C1QC', 'A2M', 'SDS', 'PLTP', 'MRC1L1', 'SLCO2B1', 'LTC4S', 'CD9', 'FCGR2B', 'RGS1', 'VSIG4', 'FPR3', 'C1QB', 'GPR34', 'AXL', 'FILIP1L', 'PKIB', 'C1QA', 'CD72', 'CLEC9A', 'NDRG2', 'CLEC10A', 'GATM', 'ALB', 'GPR183', 'MSR1', 'HLA-DPA1', 'PAPOLG', 'DAB2', 'PLD4', 'LPAR6', 'PHLDA1', 'EBI3', 'F13A1', 'FCER1A', 'HLA-DQA1', 'CPM', 'HLA-DPB1', 'ALOX5AP', 'SERPINF1', 'HLA-DRB1', 'RCAN1', 'STMN1', 'HLA-DRA', 'CCL3L1', 'ATP1B1', 'HLA-DQB1', 'PRDM1', 'MAF', 'ALCAM', 'C1orf54', 'APOC1', 'PTMS', 'LGALS3BP', 'CLIC2', 'RGCC', 'SLC40A1', 'VMO1', 'SERPING1', 'EPB41L2', 'CCL3', 'FOLR2', 'CD1C', 'PDK4', 'SEPP1', 'APOE'],
    "moKC_genes":['RNASE1', 'IGSF21', 'FCGBP', 'LILRB5', 'SLCO2B1', 'C1QC', 'SEPP1', 'PLTP', 'MRC1L1', 'NRP1', 'A2M', 'C1QB', 'TMEM37', 'LYVE1', 'FRMD4A', 'GFRA2', 'APOE', 'FOLR2', 'GPR34', 'OLFML2B', 'C1QA', 'LGMN', 'MAF', 'ID3', 'CXCL12', 'AXL', 'MERTK', 'SLC40A1', 'DAB2', 'MRC1', 'DHRS3', 'PDK4', 'GATM', 'LTC4S', 'MS4A4A', 'GPNMB', 'APOC1', 'USP53', 'EBI3', 'FPR3', 'FILIP1L', 'ALB', 'STAB1', 'SDC3', 'SIGLEC1', 'KCNMA1', 'VSIG4', 'ANKH', 'PKIB', 'MSR1', 'VCAM1', 'FCHO2', 'CCL3', 'SERPING1', 'CLEC9A', 'LGALS3BP', 'CCL4', 'CCL3L1', 'CD72', 'STMN1', 'CPM', 'CD163', 'RGL1', 'ABCA1', 'CD5L', 'EPS8', 'C2', 'CLIC2', 'RGS1', 'SPRED1', 'ATP1B1', 'TSPAN4', 'RCAN1', 'LIPA', 'SMPDL3A', 'WWP1', 'FRMD4B', 'EPB41L2', 'CD81', 'TCF4', 'AP2A2', 'SPATS2L', 'CD84', 'DST', 'FMNL2', 'MS4A7', 'CD59', 'CTSL', 'PLD3', 'HLA-DPA1', 'PTMS', 'ARL4C', 'VMO1', 'C1orf54', 'EGR2', 'CLEC10A', 'FCGR3A'],
    "KC_genes":['TIMD4', 'SLC16A9', 'NDST3', 'KCNAB1', 'ITGAD', 'CD5L', 'VCAM1', 'SELENBP1', 'BCAM', 'CDH5', 'CETP', 'RND3', 'FEZ1', 'FABP3', 'SDC3', 'CXCL12', 'ITLN1', 'CTD-2337J16.1', 'LYVE1', 'SEPP1', 'TMEM37', 'MARCO', 'SCD', 'LILRB5', 'PPAP2B', 'ITGA9', 'IGF1', 'APOE', 'SLC7A8', 'EPAS1', 'SLC40A1', 'FRMD4A', 'SUCNR1', 'ACVRL1', 'IGFBP4', 'ME1', 'LGMN', 'SIGLEC1', 'SIGLEC11', 'GPR126', 'FOLR2', 'LAG3', 'ITGB5', 'NRP1', 'C1QB', 'AXL', 'MRC1', 'C1QA', 'IFI27', 'C1QC', 'C2', 'SLC46A1', 'SLCO2B1', 'ETV5', 'SCN1B', 'NR1H3', 'DHRS3', 'PDK4', 'CD209', 'CCL4L1', 'GPNMB', 'ARHGEF12', 'CD59', 'RGL1', 'ANKRD36C', 'CCL4', 'MAF', 'EPB41L2', 'GATM', 'CCND1', 'CTSF', 'IFT74', 'LIPA', 'A2M', 'MYO9A', 'ATP1B1', 'CCL4L2', 'IFIT1', 'SERPING1', 'SPATS2L', 'MERTK', 'FUCA1', 'ADORA3', 'SMPDL3A', 'CCL3', 'ABCA1', 'MRC1L1', 'ANKH', 'CMKLR1', 'DST', 'PLD3', 'PLTP', 'FSCN1', 'APOC1', 'MMD', 'QPRT', 'VSIG4', 'RHOBTB3', 'EXT1', 'DAB2', 'ALB', 'RCAN1', 'FAM213A', 'EBI3', 'GFRA2', 'CD82', 'CCL3L3', 'PLA2G15', 'WWP1', 'CD81', 'ADM', 'ABCG1', 'FXYD6', 'STOM', 'KCNMA1', 'IFIT2', 'CD38', 'PPP1R12B', 'APOC3', 'CTSL', 'LGALS3BP', 'CR1', 'MSR1', 'TNFRSF21', 'CD163', 'GCNT1', 'EPHX1', 'SASH1', 'FILIP1L', 'SNX24', 'CLIC2', 'IFIT3', 'C1orf54', 'FPR3', 'FRMD4B', 'LTC4S', 'MMP19', 'CREG1', 'HMOX1', 'ZNF189', 'SDPR', 'NCEH1', 'FMNL2', 'CCL3L1', 'ITGAV', 'TSPAN4', 'ALDH1A1', 'NFIA', 'CTSB', 'PDE4DIP', 'ARL4C', 'ARRDC3', 'STMN1', 'RBP7', 'MS4A4A', 'EGR2', 'CD84', 'TCF4', 'VMO1', 'DSC2', 'GPR34', 'FCHO2', 'SLC9A9', 'PLA2G7', 'ADAP2', 'SCARB2', 'IL10', 'FCGRT', 'PHACTR2', 'AP2A2', 'ELL2', 'ACP2', 'MS4A7', 'P4HA1', 'PDGFC', 'GPR137B', 'SGPP1', 'SPRED1', 'PKIB', 'IFI44L', 'PRNP', 'CD72', 'SIAH2', 'MS4A6A', 'EPS8', 'UGCG', 'IDH1', 'BLVRB', 'ICAM1', 'RGS1', 'CTSD', 'CD63', 'FCGR3A', 'MT1F', 'GIMAP5', 'KCNJ2']
    }


In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["cDC1_genes"] , score_name="cDC1_score")
sc.pl.umap(adata_scmono, color='cDC1_score', cmap='bwr', title='cDC1_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["cDC2_genes"] , score_name="cDC2_score")
sc.pl.umap(adata_scmono, color='cDC2_score', cmap='bwr', title='cDC2_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["MigcDC_genes"] , score_name="MigcDC_score")
sc.pl.umap(adata_scmono, color='MigcDC_score', cmap='bwr', title='MigcDC_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["Monocyte_genes"] , score_name="Monocyte_score")
sc.pl.umap(adata_scmono, color='Monocyte_score', cmap='bwr', title='Monocyte_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["Patmono_genes"] , score_name="Patmono_score")
sc.pl.umap(adata_scmono, color='Patmono_score', cmap='bwr', title='Patmono_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["LAM_genes"] , score_name="LAM_score")
sc.pl.umap(adata_scmono, color='LAM_score', cmap='bwr', title='LAM_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["MATLAM_genes"] , score_name="MATLAM_score")
sc.pl.umap(adata_scmono, color='MATLAM_score', cmap='bwr', title='MATLAM_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["MAC1_genes"] , score_name="MAC1_score")
sc.pl.umap(adata_scmono, color='MAC1_score', cmap='bwr', title='MAC1_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["moKC_genes"] , score_name="moKC_score")
sc.pl.umap(adata_scmono, color='moKC_score', cmap='bwr', title='moKC_score')

In [ ]:
sc.tl.score_genes(adata_scmono, gene_list=gene_sets["KC_genes"] , score_name="KC_score")
sc.pl.umap(adata_scmono, color='KC_score', cmap='bwr', title='KC_score')

In [ ]:
adata_scmono

In [ ]:
adata_scmono.raw

In [ ]:
sc.tl.leiden(adata_scmono, resolution=2, key_added="myl_cluster")
sc.pl.umap(adata_scmono, color=['myl_cluster'])

In [ ]:
sc.tl.leiden(adata_scmono, resolution=0.57, key_added="10_cluster")
sc.pl.umap(adata_scmono, color=['10_cluster'])

In [ ]:
celltypist.models.models_description()

In [ ]:
predictions_cluster = celltypist.annotate(adata_scmono, model='Healthy_Human_Liver.pkl', majority_voting=True, over_clustering=adata_scmono.obs["myl_cluster"]
)
adata_scmono =predictions_cluster.to_adata()
adata_scmono.obs['celltypist_liver'] = adata_scmono.obs['majority_voting']

In [ ]:
predictions_cluster = celltypist.annotate(adata_scmono, model='Immune_All_Low.pkl', majority_voting=True, over_clustering=adata_scmono.obs["myl_cluster"]
)
adata_scmono =predictions_cluster.to_adata()

In [ ]:
adata_scmono.obs

In [ ]:
adata_scmono.obs['Immune_low'] = adata_scmono.obs['majority_voting']

In [ ]:
predictions_cluster = celltypist.annotate(adata_scmono, model='Immune_All_High.pkl', majority_voting=True, over_clustering=adata_scmono.obs["myl_cluster"]
)
adata_scmono =predictions_cluster.to_adata()

In [ ]:
adata_scmono.obs['Immune_High'] = adata_scmono.obs['majority_voting']

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='Immune_low',
    #legend_loc='on data',
    legend_fontsize=12,
    frameon=False,
    title='Immune_celltypist',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='celltypist_liver',
    #legend_loc='on data',
    legend_fontsize=12,
    frameon=False,
    title='Immune_celltypist',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='Immune_High',
    #legend_loc='on data',
    legend_fontsize=12,
    frameon=False,
    title='Immune_celltypist',
    size=10
)

In [ ]:
adata_scmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='SLC7A5',
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    title='MKI67_expression',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scmono,
    color=['CD4'],
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    title='MKI67_expression',
    size=10
)

# celltypist immune low analysis

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_scmono.obs['Immune_low']
batch_labels = adata_scmono.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
sc.pl.dotplot(adata_scmono,["SLC7A5","CCL2","NLRP3","TLR4"], groupby="Immune_low", standard_scale=None)

In [ ]:
sc.pl.dotplot(adata_scmono,["SLC7A5","CCL2","NLRP3","TLR4"], groupby="myl_cluster", standard_scale=None)

In [ ]:
adata_scmono

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'SLC7A5',groupby='myl_cluster',ax=ax)
plt.show()

In [ ]:
genes = ['SLC7A5', 'CCL2', 'NLRP3', 'TLR4', 'TNF', 'MTOR', 'AKT1', 'SREBF1']  # 예시 유전자 리스트
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(8, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_scmono,
        keys=gene,
        groupby='Immune_low',
        ax=axs[i],
        #palette = palette,
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
sc.tl.dendrogram(adata_scmono,groupby='Immune_low')
sc.tl.rank_genes_groups(adata_scmono, groupby='Immune_low', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scmono, n_genes=10)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(adata_scmono, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scmono,
    var_names=top_genes,
    groupby="Immune_low",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df

In [ ]:
adata_scmono.obs['Immune_low'].value_counts()

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 100
groups = adata_scmono.obs['Immune_low'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata_scmono.obs[adata_scmono.obs['Immune_low'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata_scmono[selected_indices]

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 100
groups = adata_scmono.obs['Immune_low'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata_scmono.obs[adata_scmono.obs['Immune_low'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata_scmono[selected_indices]
fig_dict = sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="Immune_low",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False,
    return_fig=True
)

print(type(fig_dict))
print(fig_dict.keys())

In [ ]:
fig_dict = sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="Immune_low",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False,
    return_fig=True
)

print(type(fig_dict))
print(fig_dict.keys())

In [ ]:
adata_scmono

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata_scmono.obs['state']
cluster = adata_scmono.obs['Immune_low']

# 클러스터(세포타입) 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):  # 문자형 label일 경우 그냥 sorted() 사용
    # 각 조건별 개수 계산
    donor_in = ((state == "healthy") & (cluster == clus)).sum()
    donor_out = ((state == "healthy") & (cluster != clus)).sum()
    ipf_in = ((state == "NAFLD") & (cluster == clus)).sum()
    ipf_out = ((state == "NAFLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out) if (donor_in + donor_out) > 0 else 0
    ipf_frac = ipf_in / (ipf_in + ipf_out) if (ipf_in + ipf_out) > 0 else 0
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 기준 정렬 (선택사항)
df_result = df_result.sort_values("p_value")
print(df_result)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(15, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='healthy')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.39, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata_scmono.obs['state']
cluster = adata_scmono.obs['Immune_High']

# 클러스터(세포타입) 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):  # 문자형 label일 경우 그냥 sorted() 사용
    # 각 조건별 개수 계산
    donor_in = ((state == "healthy") & (cluster == clus)).sum()
    donor_out = ((state == "healthy") & (cluster != clus)).sum()
    ipf_in = ((state == "NAFLD") & (cluster == clus)).sum()
    ipf_out = ((state == "NAFLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out) if (donor_in + donor_out) > 0 else 0
    ipf_frac = ipf_in / (ipf_in + ipf_out) if (ipf_in + ipf_out) > 0 else 0
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 기준 정렬 (선택사항)
df_result = df_result.sort_values("p_value")
print(df_result)

import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(11, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='healthy')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.5, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata_scmono.obs['state']
cluster = adata_scmono.obs['celltypist_liver']

# 클러스터(세포타입) 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):  # 문자형 label일 경우 그냥 sorted() 사용
    # 각 조건별 개수 계산
    donor_in = ((state == "healthy") & (cluster == clus)).sum()
    donor_out = ((state == "healthy") & (cluster != clus)).sum()
    ipf_in = ((state == "NAFLD") & (cluster == clus)).sum()
    ipf_out = ((state == "NAFLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out) if (donor_in + donor_out) > 0 else 0
    ipf_frac = ipf_in / (ipf_in + ipf_out) if (ipf_in + ipf_out) > 0 else 0
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 기준 정렬 (선택사항)
df_result = df_result.sort_values("p_value")
print(df_result)

import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(11, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='healthy')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.66, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

# sub clustering 10 analysis

In [ ]:
adata_scmono

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='10_cluster',
    #legend_loc='on data',
    legend_fontsize=14,
    frameon=False,
    title='subclustering_10',
    size=10
)

In [ ]:
# 2. crosstab 생성
cluster_labels = adata_scmono.obs['10_cluster']
batch_labels = adata_scmono.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_celltypist_IPF_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'SLC7A5',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'PIK3CA',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'AKT1',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'MTOR',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'PTEN',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'RHEB',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
sc.pl.violin(adata_scmono, keys = 'CTNNB1',groupby='10_cluster',ax=ax)
plt.show()

In [ ]:
sc.tl.dendrogram(adata_scmono,groupby='10_cluster')
sc.tl.rank_genes_groups(adata_scmono, groupby='10_cluster', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scmono, n_genes=10)

rank_df = sc.get.rank_genes_groups_df(adata_scmono, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scmono,
    var_names=top_genes,
    groupby="10_cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
rank_df.to_csv("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scmono_DEG.csv")

In [ ]:
sc.tl.dendrogram(adata_scmono,groupby='10_cluster')
sc.tl.rank_genes_groups(adata_scmono, groupby='10_cluster', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scmono, n_genes=10)

rank_df = sc.get.rank_genes_groups_df(adata_scmono, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scmono,
    var_names=top_genes,
    groupby="10_cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:

from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata_scmono.obs['state']
cluster = adata_scmono.obs['10_cluster']

# 클러스터 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters, key=int):  # 클러스터 번호 순 정렬
    # 각 조건별 개수 계산
    donor_in = ((state == "healthy") & (cluster == clus)).sum()
    donor_out = ((state == "healthy") & (cluster != clus)).sum()
    ipf_in = ((state == "NAFLD") & (cluster == clus)).sum()
    ipf_out = ((state == "NAFLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out)
    ipf_frac = ipf_in / (ipf_in + ipf_out)
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 정렬
df_result = df_result.sort_values("cluster")
print(df_result)
# cluster 열을 숫자형으로 바꾼 뒤 정렬
df_result['cluster'] = df_result['cluster'].astype(int)
df_result = df_result.sort_values('cluster')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(11, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='healthy')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], 0.34, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
adata_scmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")

In [ ]:
sc.pl.dotplot(adata_scmono, ['SLC7A5','PIK3CA','PIK3CB','PIK3CD','AKT1','MTOR','PTEN','RHEB','INPP4A','INPP4B'], 
              groupby="10_cluster",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['CD14','CCR2','S100A8','ITGAX','CLEC9A','CD1C','FLT3','CLEC4F','VSIG4','TIMD4'], 
              groupby="10_cluster",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['CLEC9A','FLT3','XCR1','CD1C','IRF4','CLEC10A'], 
              groupby="10_cluster",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['CLEC4F', 'VSIG4', 'MARCO', 'TIMD4', 'CD163',  'CCR2', 'S100A8', 'S100A9', 'FCN1',
                            'MARCO', 'TIMD4', 'CD163', 'CXCL9', 'CXCL10'], 
              groupby="10_cluster",
              standard_scale='var',
              figsize=(7, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['CCR2','S100A8','S100A9','CD14','CLEC4F','VSIG4','MARCO','TIMD4'], 
              groupby="10_cluster",
              standard_scale=None,
              figsize=(7, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['WNT1','FZD1','LRP6','DVL2','AXIN1','APC','GSK3B','CTNNB1','LEF1'], 
              groupby="10_cluster",
              standard_scale='var',
              figsize=(4, 3))

In [ ]:
sc.pl.dotplot(adata_scmono, ['MAP1LC3B2','MAP1LC3A','COL1A1','FN1','SLC7A5'], 
              groupby="10_cluster",
              standard_scale=None,
              figsize=(4, 3))

In [ ]:
adata_scmono.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")

In [ ]:
adata_scmono=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph

In [ ]:
adata_scmono=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_02.h5ad")
adata_scrna

In [ ]:
adata_scrna.obs['celltype'].value_counts()

In [ ]:
adata_scrna.obs['detail_cluster'] = adata_scrna.obs['celltype'].astype(str)

In [ ]:
for cell, sub in adata_scmono.obs["10_cluster"].items():
    adata_scrna.obs.at[cell, "detail_cluster"] = f"monoMAC-{sub}"

In [ ]:
print(adata_scrna.obs['detail_cluster'].value_counts())

In [ ]:
adata_scrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_monoMAC_detail.h5ad")

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_monoMAC_detail.h5ad")
adata_scrna

In [ ]:
adata_scrna.obs.to_csv("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_monoMAC_detail_obs.csv")

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='detail_cluster',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    #title='Cell type prediction',
    size=10
)

In [ ]:
umap_df = pd.DataFrame(
    adata_scrna.obsm["X_umap"],
    index=adata_scrna.obs_names,
    columns=["UMAP_1", "UMAP_2"]
)

In [ ]:
umap_df

In [ ]:
adata_scmono

In [ ]:
pd.crosstab(adata_scmono.obs['state'],adata_scmono.obs['10_cluster'])

In [ ]:
sc.pl.umap(
    adata_scmono,
    color='10_cluster',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    #title='Cell type prediction',
    size=10
)